# Protein Function Classifier in Google Colab

Train a PyTorch classifier on UniProt protein sequences. The notebook automatically uses a Colab GPU when available and switches to mixed precision on CUDA for faster training.

In [ ]:
!pip -q install requests torch

import json
import random
import re
from contextlib import nullcontext
from dataclasses import dataclass
from pathlib import Path

import requests
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

UNIPROT_SEARCH_URL = 'https://rest.uniprot.org/uniprotkb/search'
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'
AMINO_ACID_TO_INDEX = {amino_acid: index + 1 for index, amino_acid in enumerate(AA_ALPHABET)}
UNKNOWN_INDEX = len(AMINO_ACID_TO_INDEX) + 1
PAD_INDEX = 0

@dataclass(frozen=True)
class ProteinSample:
    sequence: str
    label: int


def make_session():
    session = requests.Session()
    session.headers.update({'User-Agent': 'sequenceview-protein-classifier/1.0', 'Accept': 'text/plain'})
    return session


def parse_fasta(text):
    sequences, current = [], []
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        if line.startswith('>'):
            if current:
                sequences.append(''.join(current))
                current = []
            continue
        current.append(line)
    if current:
        sequences.append(''.join(current))
    return sequences


def next_link(link_header):
    if not link_header:
        return None
    match = re.search(r'<([^>]+)>;\s*rel="next"', link_header)
    return match.group(1) if match else None


def download_uniprot_sequences(query, limit, batch_size=500, session=None):
    if limit <= 0:
        return []
    local_session = session or make_session()
    sequences = []
    url = UNIPROT_SEARCH_URL
    params = {'query': query, 'format': 'fasta', 'size': min(batch_size, limit)}
    while url and len(sequences) < limit:
        response = local_session.get(url, params=params, timeout=90)
        response.raise_for_status()
        sequences.extend(parse_fasta(response.text))
        url = next_link(response.headers.get('Link'))
        params = None
    return sequences[:limit]


def clean_sequence(sequence):
    allowed = set(AA_ALPHABET)
    return ''.join(character if character in allowed else 'X' for character in sequence.upper())


def save_dataset(samples, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as handle:
        for sample in samples:
            handle.write(json.dumps({'sequence': sample.sequence, 'label': sample.label}) + '\n')


def load_dataset(path):
    samples = []
    with path.open('r', encoding='utf-8') as handle:
        for raw_line in handle:
            record = json.loads(raw_line)
            samples.append(ProteinSample(str(record['sequence']), int(record['label'])))
    return samples


def build_labeled_dataset(enzyme_count, non_enzyme_count, cache_path=None, refresh=False):
    if cache_path and cache_path.exists() and not refresh:
        return load_dataset(cache_path)
    session = make_session()
    enzyme_query = 'reviewed:true AND ec:* AND length:[50 TO 1000]'
    non_enzyme_query = 'reviewed:true AND NOT ec:* AND length:[50 TO 1000]'
    enzyme_sequences = download_uniprot_sequences(enzyme_query, enzyme_count, session=session)
    non_enzyme_sequences = download_uniprot_sequences(non_enzyme_query, non_enzyme_count, session=session)
    samples = [ProteinSample(clean_sequence(sequence), 1) for sequence in enzyme_sequences]
    samples.extend(ProteinSample(clean_sequence(sequence), 0) for sequence in non_enzyme_sequences)
    random.shuffle(samples)
    if cache_path:
        save_dataset(samples, cache_path)
    return samples


def split_dataset(samples, train_fraction=0.8, validation_fraction=0.1, seed=13):
    grouped = {0: [], 1: []}
    for sample in samples:
        grouped[sample.label].append(sample)
    rng = random.Random(seed)
    splits = {'train': [], 'validation': [], 'test': []}
    for label_samples in grouped.values():
        rng.shuffle(label_samples)
        total = len(label_samples)
        train_end = int(total * train_fraction)
        validation_end = train_end + int(total * validation_fraction)
        splits['train'].extend(label_samples[:train_end])
        splits['validation'].extend(label_samples[train_end:validation_end])
        splits['test'].extend(label_samples[validation_end:])
    rng.shuffle(splits['train'])
    rng.shuffle(splits['validation'])
    rng.shuffle(splits['test'])
    return splits['train'], splits['validation'], splits['test']


def encode_sequence(sequence, max_length):
    encoded = [AMINO_ACID_TO_INDEX.get(character, UNKNOWN_INDEX) for character in sequence[:max_length]]
    if len(encoded) < max_length:
        encoded.extend([PAD_INDEX] * (max_length - len(encoded)))
    return encoded


class ProteinSequenceDataset(Dataset):
    def __init__(self, samples, max_length):
        self.samples = samples
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        encoded = torch.tensor(encode_sequence(sample.sequence, self.max_length), dtype=torch.long)
        label = torch.tensor(sample.label, dtype=torch.long)
        return encoded, label


class ProteinClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, conv_dim=256, gru_hidden=192, dropout=0.35):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_INDEX)
        self.encoder = nn.Sequential(
            nn.Conv1d(embedding_dim, conv_dim, kernel_size=7, padding=3),
            nn.BatchNorm1d(conv_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(conv_dim, conv_dim, kernel_size=5, padding=2),
            nn.BatchNorm1d(conv_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(conv_dim, conv_dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(conv_dim),
            nn.GELU(),
        )
        self.gru = nn.GRU(conv_dim, gru_hidden, num_layers=2, batch_first=True, bidirectional=True, dropout=dropout)
        self.head = nn.Sequential(
            nn.Linear(conv_dim * 2 + gru_hidden * 2, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 2),
        )

    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        encoded = self.encoder(embeddings.transpose(1, 2))
        conv_pool = torch.cat([torch.amax(encoded, dim=2), torch.mean(encoded, dim=2)], dim=1)
        gru_input = encoded.transpose(1, 2)
        _, hidden = self.gru(gru_input)
        gru_pool = torch.cat([hidden[-2], hidden[-1]], dim=1)
        features = torch.cat([conv_pool, gru_pool], dim=1)
        return self.head(features)


def make_dataloader(samples, max_length, batch_size, shuffle):
    return DataLoader(
        ProteinSequenceDataset(samples, max_length),
        batch_size=batch_size,
        shuffle=shuffle,
        pin_memory=torch.cuda.is_available(),
    )


def evaluate(model, data_loader, device):
    model.eval()
    loss_function = nn.CrossEntropyLoss()
    total_loss = 0.0
    total_correct = 0.0
    total_examples = 0
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs = inputs.to(device, non_blocking=torch.cuda.is_available())
            labels = labels.to(device, non_blocking=torch.cuda.is_available())
            logits = model(inputs)
            loss = loss_function(logits, labels)
            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (torch.argmax(logits, dim=1) == labels).sum().item()
            total_examples += batch_size
    return {'loss': total_loss / max(total_examples, 1), 'accuracy': total_correct / max(total_examples, 1)}


def train_model(model, train_loader, validation_loader, device, epochs, learning_rate):
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(epochs, 1))
    loss_function = nn.CrossEntropyLoss(label_smoothing=0.05)
    use_amp = device.type == 'cuda'
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        running_correct = 0.0
        running_examples = 0
        for inputs, labels in train_loader:
            inputs = inputs.to(device, non_blocking=use_amp)
            labels = labels.to(device, non_blocking=use_amp)
            optimizer.zero_grad(set_to_none=True)
            autocast_context = torch.autocast(device_type='cuda', dtype=torch.float16) if use_amp else nullcontext()
            with autocast_context:
                logits = model(inputs)
                loss = loss_function(logits, labels)
            if use_amp:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            batch_size = labels.size(0)
            running_loss += loss.item() * batch_size
            running_correct += (torch.argmax(logits, dim=1) == labels).sum().item()
            running_examples += batch_size
        scheduler.step()
        train_metrics = {'loss': running_loss / max(running_examples, 1), 'accuracy': running_correct / max(running_examples, 1)}
        validation_metrics = evaluate(model, validation_loader, device)
        print(f"epoch {epoch + 1}/{epochs} train_loss={train_metrics['loss']:.4f} train_accuracy={train_metrics['accuracy']:.4f} validation_loss={validation_metrics['loss']:.4f} validation_accuracy={validation_metrics['accuracy']:.4f}")


def predict_sequence(model, sequence, max_length, device):
    model.eval()
    encoded = torch.tensor([encode_sequence(clean_sequence(sequence), max_length)], dtype=torch.long, device=device)
    with torch.no_grad():
        probabilities = torch.softmax(model(encoded), dim=1)[0]
    enzyme_probability = probabilities[1].item()
    non_enzyme_probability = probabilities[0].item()
    return {
        'prediction': 'enzyme' if enzyme_probability >= non_enzyme_probability else 'non_enzyme',
        'enzyme_probability': enzyme_probability,
        'non_enzyme_probability': non_enzyme_probability,
    }

try:
    from google.colab import drive
    drive.mount('/content/drive')
    checkpoint_path = Path('/content/drive/MyDrive/protein_classifier.pt')
except Exception:
    checkpoint_path = Path('/content/protein_classifier.pt')

enzyme_count = 750
non_enzyme_count = 750
max_length = 768
batch_size = 64
epochs = 8
learning_rate = 7.5e-4
seed = 13
data_cache = Path('/content/uniprot_binary_dataset.jsonl')

random.seed(seed)
torch.manual_seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
print(f"cuda_available={torch.cuda.is_available()}")

samples = build_labeled_dataset(enzyme_count, non_enzyme_count, cache_path=data_cache)
train_samples, validation_samples, test_samples = split_dataset(samples, seed=seed)
train_loader = make_dataloader(train_samples, max_length, batch_size, shuffle=True)
validation_loader = make_dataloader(validation_samples, max_length, batch_size, shuffle=False)
test_loader = make_dataloader(test_samples, max_length, batch_size, shuffle=False)
model = ProteinClassifier(vocab_size=len(AMINO_ACID_TO_INDEX) + 2)
print(f"parameters={sum(parameter.numel() for parameter in model.parameters()):,}")
train_model(model, train_loader, validation_loader, device=device, epochs=epochs, learning_rate=learning_rate)
test_metrics = evaluate(model, test_loader, device)
print(test_metrics)
torch.save({'model_state_dict': model.state_dict(), 'max_length': max_length, 'vocab_size': len(AMINO_ACID_TO_INDEX) + 2}, checkpoint_path)
print(checkpoint_path)
print(predict_sequence(model, 'MVLSPADKTNVKAA', max_length, device))